# Candidate SSE Socio-Geodemographic Association

This notebook tests whether socio-geodemographic variables are associated with being a candidate SSE node. It keeps data preparation explicit in the notebook and delegates model fitting, odds-ratio tables, Wald tests, and fit statistics to `sse_detection.lib` (`sselib`).

Two complementary analysis families are fitted:

1. **Composition models** use sequence-window rows from `scotland_clustering_analysis_dataset.parquet`, joined to node-level candidate status from `node_stats`. These ask whether sequences in candidate nodes differ in age, sex, SIMD, urban/rural class, or health board composition from eligible background nodes.
2. **Node-level diversity/mixing models** use node-level entropy z-scores from `node_stats`. These ask whether candidate nodes are more or less mixed than expected for a node of the same size in the same window.

Eligible background nodes are restricted to `cluster_size >= min(candidate cluster size)`, so candidates are not compared against all singletons.

In [ ]:
from pathlib import Path
import importlib
from typing import Any
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import data as ld  # noqa: E402
from sse_detection import lib as sselib  # noqa: E402E402

# Reload local package after editing source files
importlib.reload(sselib)
importlib.reload(ld)

OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "sse_outputs"
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "association_outputs"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Model Specification

Primary models adjust for the identifiable time/variant structure:

- `C(window_idx)`
- `C(clade)` by default, via `VARIANT_ADJUSTER`

The window-level surveillance summaries (`wn_prop_sequenced`, `wn_positive_tests`) are documented below as an optional no-window-fixed-effect sensitivity specification. They are not included in the main primary models because they are constants within `window_idx`, so adding them alongside `C(window_idx)` creates rank collinearity and can prevent convergence.

Expanded composition models add each sequence's own data-zone surveillance and epidemic-burden context. Expanded node-level models add node-level cluster means of the same data-zone context variables.


In [ ]:
VARIANT_ADJUSTER = "clade"  # change to "who_voc" for a coarser variant adjustment
CLUSTER_SE = "meta_cluster_id"  # robust SE clustering unit

PRIMARY_COMPOSITION_ADJUSTERS = [
    "C(window_idx)",
    f"C({VARIANT_ADJUSTER})",
]

WINDOW_SURVEILLANCE_ADJUSTERS = [
    "wn_prop_sequenced",
    "np.log1p(wn_positive_tests)",
]

# Optional sensitivity model if you want window-level surveillance adjustment
# instead of window fixed effects. Do not combine these with C(window_idx).
SURVEILLANCE_COMPOSITION_ADJUSTERS = [
    f"C({VARIANT_ADJUSTER})",
    *WINDOW_SURVEILLANCE_ADJUSTERS,
]

EXPANDED_CONTEXT_ADJUSTERS = [
    "dz_cum_prop_sequenced",
    "dz_cum_incidence_per_capita",
    "dz_7d_test_positivity",
    "np.log1p(dz_cum_positive_tests)",
]

EXPANDED_COMPOSITION_ADJUSTERS = (
    PRIMARY_COMPOSITION_ADJUSTERS + EXPANDED_CONTEXT_ADJUSTERS
)

PRIMARY_MIXING_ADJUSTERS = [
    "C(window_idx)",
    f"C({VARIANT_ADJUSTER})",
]

EXPANDED_MIXING_ADJUSTERS = (
    PRIMARY_MIXING_ADJUSTERS + EXPANDED_CONTEXT_ADJUSTERS
)

COMPOSITION_MODEL_SETS = {
    "primary": PRIMARY_COMPOSITION_ADJUSTERS,
    "expanded": EXPANDED_COMPOSITION_ADJUSTERS,
    # "surveillance_no_window_fe": SURVEILLANCE_COMPOSITION_ADJUSTERS,
}

MIXING_MODEL_SETS = {
    "primary": PRIMARY_MIXING_ADJUSTERS,
    "expanded": EXPANDED_MIXING_ADJUSTERS,
}

COMPOSITION_SPECS = [
    {
        "name": "sex",
        "column": "sex",
        "reference": "Male",
        "label": "Sex",
    },
    {
        "name": "age_band",
        "column": "age_band",
        "reference": "30-34",
        "fallback_references": ["35-39", "25-29"],
        "label": "Age band",
    },
    {
        "name": "simd_quintile",
        "column": "dz_simd_quintile",
        "reference": "3",
        "label": "SIMD quintile",
    },
    {
        "name": "urban_rural_class",
        "column": "dz_urban_rural_class",
        "reference": "Large Urban Areas",
        "label": "Urban/rural class",
    },
    {
        "name": "health_board",
        "column": "dz_health_board",
        "reference": "Greater Glasgow and Clyde",
        "label": "Health board",
    },
]

MIXING_FEATURES = [
    "sex_entropy_z",
    "age_entropy_z",
    "simd_entropy_z",
    "urban_rural_entropy_z",
    "health_board_entropy_z",
]

## Load and Prepare Analysis Frames

`node_stats` is the source of the candidate label and the node-level mixing metrics. For composition models, sequence-window rows are loaded from the processed analysis dataset using `utils.data`, filtered to match `sse_detection.ipynb`'s retained odd windows, renumbered, and then joined to eligible node status.

In [ ]:
outs = sselib.load_sse_outputs(OUTPUT_DIR)
node_stats = outs.node_stats.copy()

min_candidate_size = int(node_stats.loc[node_stats["sse_candidate"], "cluster_size"].min())
eligible_nodes = node_stats.loc[node_stats["cluster_size"].ge(min_candidate_size)].copy()

if CLUSTER_SE not in eligible_nodes.columns:
    raise KeyError(f"{CLUSTER_SE!r} is not present in node_stats.")
if VARIANT_ADJUSTER not in eligible_nodes.columns:
    raise KeyError(f"{VARIANT_ADJUSTER!r} is not present in node_stats.")

# Node-level frame for mixing models.
node_model_base = eligible_nodes.copy()
node_model_base["candidate"] = node_model_base["sse_candidate"].astype(int)
node_model_base[VARIANT_ADJUSTER] = node_model_base[VARIANT_ADJUSTER].fillna("Missing").astype(str)
node_model_base[CLUSTER_SE] = node_model_base[CLUSTER_SE].fillna(node_model_base["cluster_id"]).astype(str)

node_key = (
    eligible_nodes[["cluster_id", "sse_candidate", CLUSTER_SE, "cluster_size"]]
    .drop_duplicates("cluster_id")
)

sequence_columns = sorted({
    "window_id",
    "window_idx",
    "cluster_id",
    "sequence_id",
    "clade",
    "who_voc",
    "wn_prop_sequenced",
    "wn_positive_tests",
    "dz_cum_prop_sequenced",
    "dz_cum_incidence_per_capita",
    "dz_7d_test_positivity",
    "dz_cum_positive_tests",
    *(spec["column"] for spec in COMPOSITION_SPECS),
})

sequence_raw = ld.load_analysis_columns(sequence_columns, add_policy=False)

# Match sse_detection.ipynb: keep every other original window and renumber retained windows.
sequence_raw = sequence_raw.loc[sequence_raw["window_idx"] % 2 == 1].copy()
old_to_new = {
    old: new + 1
    for new, old in enumerate(sorted(sequence_raw["window_idx"].unique()))
}
sequence_raw["window_idx"] = sequence_raw["window_idx"].map(old_to_new)
sequence_raw["window_id"] = sequence_raw["window_idx"].apply(lambda x: f"W{x:03d}")

composition_base = sequence_raw.merge(node_key, on="cluster_id", how="inner")
composition_base["candidate"] = composition_base["sse_candidate"].astype(int)
composition_base[VARIANT_ADJUSTER] = composition_base[VARIANT_ADJUSTER].fillna("Missing").astype(str)
composition_base[CLUSTER_SE] = composition_base[CLUSTER_SE].fillna(composition_base["cluster_id"]).astype(str)

print(f"node_stats: {len(node_stats):,} nodes")
print(f"candidate nodes: {node_stats['sse_candidate'].sum():,}")
print(f"minimum candidate cluster size: {min_candidate_size}")
print(f"eligible nodes: {len(eligible_nodes):,}")
print(f"eligible candidates: {eligible_nodes['sse_candidate'].sum():,}")
print(f"eligible background: {(~eligible_nodes['sse_candidate']).sum():,}")
print()
print(f"composition sequence-window rows: {len(composition_base):,}")
print(f"unique sequences in composition frame: {composition_base['sequence_id'].nunique():,}")
print(f"nodes in composition frame: {composition_base['cluster_id'].nunique():,}")
print(f"candidate share in composition frame: {composition_base['candidate'].mean():.1%}")

## Notebook-Side Preparation Helpers

The functions below deliberately handle data preparation in the notebook: complete-case filtering, reference-level resolution, and removal of tiny strata with no candidate/background variation. The regression module is then called only on already-prepared model frames.

In [ ]:
def term_prefix(term: str) -> str:
    return term.split(",")[0]


def resolve_reference(data: pd.DataFrame, column: str, preferred, fallbacks=None):
    levels = set(data[column].dropna().astype(str))
    candidates = [preferred, *(fallbacks or [])]
    for ref in candidates:
        if ref is None:
            continue
        ref_str = str(ref)
        if ref_str in levels:
            return ref_str
        for level in levels:
            try:
                if float(level) == float(ref_str):
                    return level
            except ValueError:
                pass
    counts = data[column].dropna().astype(str).value_counts()
    if counts.empty:
        raise ValueError(f"No observed levels for {column!r}.")
    return str(counts.index[0])


def complete_case(data: pd.DataFrame, required: list[str]) -> pd.DataFrame:
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    return data.dropna(subset=required).copy()


def drop_nonvarying_levels(
    data: pd.DataFrame,
    columns: list[str],
    *,
    outcome: str = "candidate",
) -> tuple[pd.DataFrame, dict[str, int]]:
    d = data.copy()
    dropped: dict[str, int] = {}
    changed = True
    while changed:
        changed = False
        for col in columns:
            if col not in d.columns:
                continue
            varies = d.groupby(col, dropna=False)[outcome].transform("nunique").gt(1)
            n_drop = int((~varies).sum())
            if n_drop:
                dropped[col] = dropped.get(col, 0) + n_drop
                d = d.loc[varies].copy()
                changed = True
    return d, dropped


def add_model_metadata(table: pd.DataFrame, **metadata) -> pd.DataFrame:
    out = table.copy()
    for key, value in reversed(list(metadata.items())):
        out.insert(0, key, value)
    return out


def add_fit_metadata(fit_stats: pd.DataFrame, **metadata) -> pd.DataFrame:
    out = fit_stats.copy()
    for key, value in reversed(list(metadata.items())):
        out.insert(0, key, value)
    return out


def bh_adjust_by(table: pd.DataFrame, group_cols: list[str], p_col: str = "P>chi2") -> pd.DataFrame:
    if table.empty:
        return table.copy()
    out = table.copy()
    out["p_adj_bh"] = np.nan
    for _, idx in out.groupby(group_cols, dropna=False).groups.items():
        adjusted = sselib.bh_adjust(out.loc[idx], p_col=p_col)
        out.loc[idx, "p_adj_bh"] = adjusted["p_adj_bh"].to_numpy()
    return out


## Composition Models

Each composition predictor is first entered **one at a time**. Then all composition predictors are entered **together**. Both sets are fitted with primary adjusters and expanded adjusters. Odds ratios are sequence-level associations with candidate-node membership, with robust standard errors clustered by `meta_cluster_id`.

In [ ]:
def prepare_composition_frame(predictors: list[str], adjusters: list[str]) -> Any:
    required = (
        ["candidate", "cluster_id", "sequence_id", CLUSTER_SE]
        + predictors
        + sselib.model_variables_from_terms(adjusters)
    )
    required = list(dict.fromkeys(required))
    d = complete_case(composition_base, required)
    for col in predictors:
        d[col] = d[col].astype(str)
    strata = ["window_idx", VARIANT_ADJUSTER, *predictors]
    d, dropped = drop_nonvarying_levels(d, strata)
    return d, dropped


def fit_single_composition_models(model_set: str, adjusters: list[str]) -> Any:
    wald_tables = []
    or_tables = []
    fit_tables = []
    fitted = {}

    for spec in COMPOSITION_SPECS:
        predictor = spec["column"]
        d, dropped = prepare_composition_frame([predictor], adjusters)
        reference = resolve_reference(
            d,
            predictor,
            spec.get("reference"),
            spec.get("fallback_references"),
        )
        model_name = f"composition__{model_set}__single__{spec['name']}"
        fit = sselib.fit_exposure_model(
            d,
            outcome="candidate",
            exposure=predictor,
            adjusters=adjusters,
            model_name=model_name,
            reference=reference,
            cluster_col=CLUSTER_SE,
            categorical=True,
        )
        fitted[spec["name"]] = fit.result

        meta = {
            "domain": "composition",
            "model_set": model_set,
            "predictor_set": "single",
            "predictor": spec["name"],
            "label": spec["label"],
            "reference": reference,
            "n_model_rows": len(d),
            "n_sequences": d["sequence_id"].nunique(),
            "n_nodes": d["cluster_id"].nunique(),
            "dropped_nonvarying_rows": sum(dropped.values()),
            "dropped_nonvarying_detail": repr(dropped),
        }
        wald_tables.append(add_model_metadata(fit.wald, **meta))
        or_tables.append(add_model_metadata(fit.odds_ratios, **meta))
        fit_tables.append(add_fit_metadata(
            sselib.model_fit_stats(fit.result, model_name=model_name, formula=fit.formula),
            **meta,
        ))
        print(f"Fitted {model_name}: {len(d):,} rows", flush=True)

    return (
        fitted, pd.concat(wald_tables, ignore_index=True), 
        pd.concat(or_tables, ignore_index=True), 
        pd.concat(fit_tables, ignore_index=True)
        )


def fit_joint_composition_model(model_set: str, adjusters: list[str]) -> Any:
    predictors = [spec["column"] for spec in COMPOSITION_SPECS]
    d, dropped = prepare_composition_frame(predictors, adjusters)

    terms = []
    references = {}
    for spec in COMPOSITION_SPECS:
        reference = resolve_reference(
            d,
            spec["column"],
            spec.get("reference"),
            spec.get("fallback_references"),
        )
        references[spec["name"]] = reference
        terms.append(sselib.categorical_term(spec["column"], reference))

    formula = "candidate ~ " + " + ".join(terms + adjusters)
    model_name = f"composition__{model_set}__joint"
    result = sselib.fit_binomial_glm(d, formula, cluster_col=CLUSTER_SE)

    wald_tables = []
    for spec, term in zip(COMPOSITION_SPECS, terms):
        wald = sselib.robust_wald_for_prefix(
            result,
            term_prefix(term),
            model_name=model_name,
            term=spec["name"],
        )
        meta = {
            "domain": "composition",
            "model_set": model_set,
            "predictor_set": "joint",
            "predictor": spec["name"],
            "label": spec["label"],
            "reference": references[spec["name"]],
            "n_model_rows": len(d),
            "n_sequences": d["sequence_id"].nunique(),
            "n_nodes": d["cluster_id"].nunique(),
            "dropped_nonvarying_rows": sum(dropped.values()),
            "dropped_nonvarying_detail": repr(dropped),
        }
        wald_tables.append(add_model_metadata(wald, **meta))

    prefixes = tuple(term_prefix(term) for term in terms)
    odds = sselib.tidy_odds_ratios(result, model_name=model_name)
    odds = odds.loc[odds["term"].str.startswith(prefixes)].copy()
    odds = add_model_metadata(
        odds,
        domain="composition",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_composition",
        label="All composition predictors",
        reference=repr(references),
        n_model_rows=len(d),
        n_sequences=d["sequence_id"].nunique(),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    fit_stats = add_fit_metadata(
        sselib.model_fit_stats(result, model_name=model_name, formula=formula),
        domain="composition",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_composition",
        label="All composition predictors",
        reference=repr(references),
        n_model_rows=len(d),
        n_sequences=d["sequence_id"].nunique(),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    print(f"Fitted {model_name}: {len(d):,} rows", flush=True)
    return result, pd.concat(wald_tables, ignore_index=True), odds, fit_stats


composition_model_results = {}
composition_fits = {}


def run_composition_model_set(model_set: str, adjusters: list[str]) -> Any:
    single_fit, single_wald, single_or, single_fit_stats = fit_single_composition_models(model_set, adjusters)
    joint_fit, joint_wald, joint_or, joint_fit_stats = fit_joint_composition_model(model_set, adjusters)
    composition_fits[(model_set, "single")] = single_fit
    composition_fits[(model_set, "joint")] = joint_fit
    composition_model_results[model_set] = {
        "wald": pd.concat([single_wald, joint_wald], ignore_index=True),
        "odds": pd.concat([single_or, joint_or], ignore_index=True),
        "fit_stats": pd.concat([single_fit_stats, joint_fit_stats], ignore_index=True),
    }
    return composition_model_results[model_set]


### Primary Composition Models

These are the main single-predictor and joint composition models adjusted for window, variant, and window-level surveillance intensity.

In [ ]:
composition_primary_results = run_composition_model_set(
    "primary",
    COMPOSITION_MODEL_SETS["primary"],
)

composition_primary_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "chi2", "df", "P>chi2", "n_model_rows", "n_sequences", "n_nodes",
    "dropped_nonvarying_detail",
]]


### Expanded Composition Models

These repeat the composition models after adding sequence data-zone surveillance and epidemic-burden context. Treat these as a robustness/sensitivity specification.

In [ ]:
composition_expanded_results = run_composition_model_set(
    "expanded",
    COMPOSITION_MODEL_SETS["expanded"],
)

composition_expanded_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "chi2", "df", "P>chi2", "n_model_rows", "n_sequences", "n_nodes",
    "dropped_nonvarying_detail",
]]


### Composition Summary Tables

The omnibus Wald table is the main screening table. McFadden pseudo-R2 is included for relative comparison within this model family.

In [ ]:
if not composition_model_results:
    raise RuntimeError("Run at least one composition model-set cell before summarising.")

composition_wald = pd.concat(
    [result["wald"] for result in composition_model_results.values()],
    ignore_index=True,
)
composition_wald = bh_adjust_by(composition_wald, ["domain", "model_set", "predictor_set"])
composition_or = pd.concat(
    [result["odds"] for result in composition_model_results.values()],
    ignore_index=True,
)
composition_fit_stats = pd.concat(
    [result["fit_stats"] for result in composition_model_results.values()],
    ignore_index=True,
)

display(composition_wald[[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "chi2", "df", "P>chi2", "p_adj_bh", "n_model_rows", "n_sequences", "n_nodes",
    "dropped_nonvarying_detail",
]])

display(composition_fit_stats[[
    "domain", "model_set", "predictor_set", "predictor", "r2_mcfadden", "converged",
    "aic", "bic_llf", "log_likelihood", "ll_null", "n_model_rows", "n_sequences", "n_nodes",
]])


### Composition Odds Ratios

The table below contains coefficient-level odds ratios for the composition models. For primary interpretation, start with the omnibus Wald table above; use this table to identify which levels drive an omnibus association.

In [ ]:
display(composition_or[[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "term", "estimate", "std_error", "p_value", "odds_ratio", "or_low", "or_high",
]].sort_values(["model_set", "predictor_set", "predictor", "p_value"]))

## Node-Level Diversity / Mixing Models

Node-level mixing models use entropy z-scores. Each coefficient is the association per one-unit increase in the entropy z-score. Primary models adjust for window and variant; expanded models additionally adjust for node-level cluster means of data-zone surveillance and burden variables.

Do **not** add candidate-defining quantities such as `cluster_size`, `core_amplification_score`, `out_strength`, or `onward_dissemination_score` to the main models, because that would condition on the machinery used to define the outcome.

In [ ]:
def prepare_mixing_frame(predictors: list[str], adjusters: list[str]) -> Any:
    required = (
        ["candidate", "cluster_id", CLUSTER_SE]
        + predictors
        + sselib.model_variables_from_terms(adjusters)
    )
    required = list(dict.fromkeys(required))
    d = complete_case(node_model_base, required)
    strata = ["window_idx", VARIANT_ADJUSTER]
    d, dropped = drop_nonvarying_levels(d, strata)
    return d, dropped


def fit_single_mixing_models(model_set: str, adjusters: list[str]) -> Any:
    wald_tables = []
    or_tables = []
    fit_tables = []
    fitted = {}

    for feature in MIXING_FEATURES:
        if feature not in node_model_base.columns:
            print(f"Skipping {feature}: not found", flush=True)
            continue
        d, dropped = prepare_mixing_frame([feature], adjusters)
        model_name = f"mixing__{model_set}__single__{feature}"
        fit = sselib.fit_exposure_model(
            d,
            outcome="candidate",
            exposure=feature,
            adjusters=adjusters,
            model_name=model_name,
            cluster_col=CLUSTER_SE,
            categorical=False,
        )
        fitted[feature] = fit.result
        meta = {
            "domain": "node_mixing",
            "model_set": model_set,
            "predictor_set": "single",
            "predictor": feature,
            "label": feature.replace("_", " "),
            "reference": "per 1 z-score",
            "n_model_rows": len(d),
            "n_nodes": d["cluster_id"].nunique(),
            "dropped_nonvarying_rows": sum(dropped.values()),
            "dropped_nonvarying_detail": repr(dropped),
        }
        wald_tables.append(add_model_metadata(fit.wald, **meta))
        or_tables.append(add_model_metadata(
            fit.odds_ratios.loc[fit.odds_ratios["term"].eq(feature)].copy(),
            **meta,
        ))
        fit_tables.append(add_fit_metadata(
            sselib.model_fit_stats(fit.result, model_name=model_name, formula=fit.formula),
            **meta,
        ))
        print(f"Fitted {model_name}: {len(d):,} nodes", flush=True)
    return (
        fitted, pd.concat(wald_tables, ignore_index=True), 
        pd.concat(or_tables, ignore_index=True), 
        pd.concat(fit_tables, ignore_index=True)
        )


def fit_joint_mixing_model(model_set: str, adjusters: list[str]) -> Any:
    features = [feature for feature in MIXING_FEATURES if feature in node_model_base.columns]
    d, dropped = prepare_mixing_frame(features, adjusters)
    formula = "candidate ~ " + " + ".join(features + adjusters)
    model_name = f"mixing__{model_set}__joint"
    result = sselib.fit_binomial_glm(d, formula, cluster_col=CLUSTER_SE)

    wald = sselib.tidy_single_parameter_wald(result, features, model_name=model_name)
    wald = add_model_metadata(
        wald,
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference="per 1 z-score",
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    odds = sselib.tidy_odds_ratios(result, model_name=model_name)
    odds = odds.loc[odds["term"].isin(features)].copy()
    odds = add_model_metadata(
        odds,
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference="per 1 z-score",
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    fit_stats = add_fit_metadata(
        sselib.model_fit_stats(result, model_name=model_name, formula=formula),
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference="per 1 z-score",
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    print(f"Fitted {model_name}: {len(d):,} nodes", flush=True)
    return result, wald, odds, fit_stats


mixing_model_results = {}
mixing_fits = {}


def run_mixing_model_set(model_set: str, adjusters: list[str]) -> Any:
    single_fit, single_wald, single_or, single_fit_stats = fit_single_mixing_models(model_set, adjusters)
    joint_fit, joint_wald, joint_or, joint_fit_stats = fit_joint_mixing_model(model_set, adjusters)
    mixing_fits[(model_set, "single")] = single_fit
    mixing_fits[(model_set, "joint")] = joint_fit
    mixing_model_results[model_set] = {
        "wald": pd.concat([single_wald, joint_wald], ignore_index=True),
        "odds": pd.concat([single_or, joint_or], ignore_index=True),
        "fit_stats": pd.concat([single_fit_stats, joint_fit_stats], ignore_index=True),
    }
    return mixing_model_results[model_set]


### Primary Mixing Models

These node-level models test each entropy z-score alone and then all entropy z-scores together, adjusted for window and variant.

In [ ]:
mixing_primary_results = run_mixing_model_set(
    "primary",
    MIXING_MODEL_SETS["primary"],
)

mixing_primary_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "chi2", "df", "P>chi2", "n_model_rows", "n_nodes",
    "dropped_nonvarying_detail",
]]


### Expanded Mixing Models

These add node-level cluster means of the data-zone context variables, matching the expanded composition sensitivity model.

In [ ]:
mixing_expanded_results = run_mixing_model_set(
    "expanded",
    MIXING_MODEL_SETS["expanded"],
)

mixing_expanded_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "chi2", "df", "P>chi2", "n_model_rows", "n_nodes",
    "dropped_nonvarying_detail",
]]


### Mixing Summary Tables

The Wald table gives the per-feature evidence. The odds-ratio table below gives the direction and magnitude per one-unit increase in the entropy z-score.

In [ ]:
if not mixing_model_results:
    raise RuntimeError("Run at least one mixing model-set cell before summarising.")

mixing_wald = pd.concat(
    [result["wald"] for result in mixing_model_results.values()],
    ignore_index=True,
)
mixing_wald = bh_adjust_by(mixing_wald, ["domain", "model_set", "predictor_set"])
mixing_or = pd.concat(
    [result["odds"] for result in mixing_model_results.values()],
    ignore_index=True,
)
mixing_fit_stats = pd.concat(
    [result["fit_stats"] for result in mixing_model_results.values()],
    ignore_index=True,
)

display(mixing_wald[[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "chi2", "df", "P>chi2", "p_adj_bh", "n_model_rows", "n_nodes",
    "dropped_nonvarying_detail",
]])

display(mixing_fit_stats[[
    "domain", "model_set", "predictor_set", "predictor", "r2_mcfadden", "converged",
    "aic", "bic_llf", "log_likelihood", "ll_null", "n_model_rows", "n_nodes",
]])


### Mixing Odds Ratios

For entropy z-score models, odds ratios are per one-unit increase in the entropy z-score. OR < 1 means candidate nodes are less likely at higher-than-null mixing; OR > 1 means candidate nodes are more likely at higher-than-null mixing.

In [ ]:
display(mixing_or[[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "estimate", "std_error", "p_value", "odds_ratio", "or_low", "or_high",
]].sort_values(["model_set", "predictor_set", "predictor", "p_value"]))

## Interpretation Guide

Use the tables in this order:

1. **Single-predictor primary models**: main adjusted association for each socio-geodemographic variable, controlling for time and variant context.
2. **Single-predictor expanded models**: sensitivity to local data-zone surveillance and epidemic-burden adjustment.
3. **Joint primary models**: whether each predictor retains signal when the socio-geodemographic predictors are mutually adjusted.
4. **Joint expanded models**: the most conditional specification; useful as a robustness check rather than the simplest effect summary.
5. **McFadden pseudo-R2**: compare models within the same outcome/family and analysis frame. Small values are normal for logistic models; the main use here is relative comparison between primary vs expanded and single vs joint models.

Composition models describe **who/where the sequences in candidate nodes come from**. Node-level mixing models describe **whether candidate nodes are unusually internally diverse or concentrated** for their size and window.

In [ ]:
summary_tables = {
    "composition_wald.csv": composition_wald,
    "composition_odds_ratios.csv": composition_or,
    "composition_fit_stats.csv": composition_fit_stats,
    "mixing_wald.csv": mixing_wald,
    "mixing_odds_ratios.csv": mixing_or,
    "mixing_fit_stats.csv": mixing_fit_stats,
}

for filename, table in summary_tables.items():
    path = RESULT_DIR / filename
    table.to_csv(path, index=False)
    print(f"saved {filename}: {len(table):,} rows")